In [1]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path
import re

In [2]:
np.random.seed(0)


In [3]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/safe_sentences_timit_wsn_compatible_0.1rms.pdpkl'
timit_excerpts = pd.read_pickle(path)


In [4]:
timit_excerpts = timit_excerpts.rename(columns={"_full_dataset_index":"_original_timit_index"})

In [5]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

# filter cues 
cue_timit = timit_excerpts[timit_excerpts.word_int == 0]
# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]


In [6]:
target_timit.reset_index(inplace=True, drop=True)

In [7]:
target_timit.shape

(902, 13)

## Define functions to be used with STRAIGHT

Use f0 shift with no actual shift to resynthesize the speech - this is effectively passing the signal through STRAIGHT, which may add processing artifacts while keeping the speech otherwise unaltered

In [12]:
import scipy.signal
import scipy.interpolate
import scipy.io.wavfile
import librosa
import librosa.display
import matlab.engine
import sys 
sys.path.append('/om2/user/msaddler/python-packages/msutil')
import util_stimuli
import util_misc

# def shift_f0_contour(y, sr, f0_shift_in_octaves, eng=None):
#     x = matlab.double(y.reshape([-1, 1]).tolist())
#     y_new_matlab, f0_old_matlab, f0_new_matlab = eng.legacy_STRAIGHT_shift_f0_contour(
#         x,
#         matlab.double([sr]),
#         matlab.double([f0_shift_in_octaves]),
#         nargout=3)
#     y_resynth = np.array(y_new_matlab).astype(np.float32).reshape([-1])
#     f0_old = np.array(f0_old_matlab).astype(np.float32).reshape([-1])
#     f0_new = np.array(f0_new_matlab).astype(np.float32).reshape([-1])
#     NAN_IDX = np.isnan(y_resynth)
#     if NAN_IDX.sum() > 0:
#         print("Replacing {} NaN values (of {}) with zeros".format(NAN_IDX.sum(), NAN_IDX.shape[0]))
#         y_resynth[NAN_IDX] = 0
#     if len(y_resynth) > 0:
#         npad = y.shape[0] - y_resynth.shape[0]
#         y_resynth = np.pad(y_resynth, [int(np.ceil(npad/2)), int(npad/2)])
#     return y_resynth, f0_old, f0_new


def make_harm_speech(y, sr, eng=None):
    x = matlab.double(y.reshape([-1, 1]).tolist())
    y_new_matlab= eng.StraightDummySynth(
        x,
        matlab.double([sr])
        )
    y_resynth = np.array(y_new_matlab).astype(np.float32).reshape([-1])
    NAN_IDX = np.isnan(y_resynth)
    if NAN_IDX.sum() > 0:
        print("Replacing {} NaN values (of {}) with zeros".format(NAN_IDX.sum(), NAN_IDX.shape[0]))
        y_resynth[NAN_IDX] = 0
    if len(y_resynth) > 0:
        npad = y.shape[0] - y_resynth.shape[0]
        y_resynth = np.pad(y_resynth, [int(np.ceil(npad/2)), int(npad/2)])
    return y_resynth
# 



In [9]:
eng = matlab.engine.start_matlab();
eng.addpath('/om2/user/imgriff/projects/msSTRAIGHT/Sinusoidal_Straight_Toolbox_v1.0/');
eng.addpath('/om2/user/imgriff/projects/msSTRAIGHT/');
eng.addpath('/om2/user/imgriff/projects/msSTRAIGHT/legacy_STRAIGHT/src/');


In [13]:
y_old = target_timit['signal'].iloc[0]

In [14]:
sr=20000

## Making Harmonic Speech Dataset 

Use only single-distractors.  We are trying to emulate the concurent sentences experiment (experiment 5) in the Popham et al. 2018 paper. The paper usees 0dB SNR so we will start with setting target and distractor to the same level (0dB). 

Proceedure for generating each set of trial stimuli:
 * select cue, target, and distractor signals
 * resynth with STRAIGHT
 * mix target and distractor 
 * save all  signals (cue, target, distractor, and mixture)
 * save f0 trajectory too 
 
 
 
 

### demo of single trial:

In [15]:
target = target_timit['signal'].iloc[0]
speaker = target_timit['speaker'].iloc[0]

cue = cue_timit[cue_timit.speaker == speaker].sample(1)['signal'].item()

distractor_data = target_timit[target_timit.speaker!=speaker].sample(1)
distractor = distractor_data['signal'].item()


cue_new  = make_harm_speech(cue, sr,  eng=eng)
target_new = make_harm_speech(target, sr,  eng=eng)
distractor_new = make_harm_speech(distractor, sr,  eng=eng)



Elapsed time is 0.498817 seconds.
Elapsed time is 0.074594 seconds.
Elapsed time is 0.381451 seconds.
Elapsed time is 0.724987 seconds.
Elapsed time is 0.186610 seconds.
Elapsed time is 0.024063 seconds.
Elapsed time is 0.091229 seconds.
Elapsed time is 0.310259 seconds.
Elapsed time is 0.221570 seconds.
Elapsed time is 0.023978 seconds.
Elapsed time is 0.116321 seconds.
Elapsed time is 0.459477 seconds.


In [34]:
y_resynth = impose_inharmonic_jitter_pattern(y, 20_000, jitter_pattern, eng=eng)

Elapsed time is 0.403606 seconds.
Elapsed time is 0.030503 seconds.
Elapsed time is 0.163040 seconds.
Elapsed time is 0.825320 seconds.


In [41]:
util_stimuli.rms(y_resynth), util_stimuli.rms(y)

(0.14777178, 0.10000000000000002)

In [17]:
from IPython.display import Audio

Audio(target_new, rate=20000, normalize=True)

In [50]:
y_whispered = make_whispered_speech(y, 20_000, eng=eng)

Elapsed time is 0.339474 seconds.
Elapsed time is 0.025442 seconds.
Elapsed time is 0.166376 seconds.
Elapsed time is 0.453419 seconds.


In [21]:
next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

## Define windowing function - will apply a cosine ramp to the start and end of a signal

In [22]:
def update_dict(dict_to_update, dict_with_vals):
    for key,value in dict_with_vals.items():
        if key not in dict_to_update:
            dict_to_update[key] = [value]
        else:
            dict_to_update[key].append(value)
            

In [23]:
row

word                                                                 asked
_original_timit_index                                                  314
source                                              train-dr1-mrso0-si2289
speaker                                                              mrso0
sr                                                                   20000
signal_length                                                        40000
speaker_sex                                                              m
sentence_type                                                           si
sentence_id                                                           2289
dialect_region                                                         dr1
data_split                                                           train
signal                   [0.0023398756, 0.00032958086, -0.000252051, 0....
word_int                                                                55
Name: 50, dtype: object

## Below is example of loop used to create stimuli

This loop takes too long to run (~8 hours). To speed things up, we parallelized stimuli generation using `src/make_inharm_timit.py` and `submit_make_inharm_timit.sh` 

To get the final dataset, concatenate the stimuli into one df

In [24]:
# Generate trial by trial stim    
trial_data = {'signal':[],
              'target_signal':[],
              'speaker': [],
              'speaker_sex': [],
              'sentence_type': [],
              'word_int': [],
              'mixture_signal':[],
              'cue_signal': [],
              'cue_speaker': [],
              'cue_word': [],
              'distractor_signal':[],
              '_original_distractor_timit_indices':[],
              '_original_cue_timit_index':[],
              'distractor_words':[],
              'distractor_speakers':[],
              'distractor_conditions':[],
              'distractor_sex':[],
              'snrs':[]}



snr = 0 # start with 0 dB 
for ix, row in tqdm(target_timit.iterrows(), total=len(target_timit)):
    # add wantned data to trial dict
    update_dict(trial_data, row)
    
    # get signals 
    target_signal = row.signal
    
    distractor_data = target_timit[target_timit.speaker!=row.speaker].sample(1)
    distractor_signal = distractor_data['signal'].item()
    
    cue_data = cue_timit[cue_timit.speaker == row.speaker].sample(1)
    cue_signal = cue_data['signal'].item()
    
    cue_new, cue_f0_old, cue_f0_new = shift_f0_contour(cue_signal, sr, f0_shift_in_octaves=0, eng=eng)
    target_new, target_f0_old, target_f0_new = shift_f0_contour(target_signal, sr, f0_shift_in_octaves=0, eng=eng)
    distractor_new, distractor_f0_old, distractor_f0_new = shift_f0_contour(distractor_signal, sr, f0_shift_in_octaves=0, eng=eng)


    mixture, _ = combine_with_noise(target_new, distractor_new, snr) # first_scale_factor unused
    mixture, mixture_scale_factor = rms_normalize(mixture)
    
    cue, _ = rms_normalize(cue_new)
    # rove cue
    cue = cue * mixture_scale_factor
    
    # save values for tiral 
    trial_data['signal'][ix] = target_new
    trial_data['mixture_signal'].append(mixture)
    trial_data['distractor_signal'].append(distractor_new)
    trial_data['cue_signal'].append(cue)
    trial_data['cue_word'].append(cue_data.word.item())
    trial_data['cue_speaker'].append(cue_data.speaker.item())
    trial_data['_original_cue_timit_index'].append(cue_data._original_timit_index.item())
    trial_data['_original_distractor_timit_indices'].append(distractor_data._original_timit_index.item())
    trial_data['distractor_words'].append(distractor_data.word.item())
    trial_data['distractor_speakers'].append(distractor_data.speaker.item())
    trial_data['distractor_conditions'].append('inharmonic')
    trial_data['distractor_sex'].append(distractor_data.speaker_sex.item())
    trial_data['snrs'].append(snr)
#     if ix == 1:
#         break


  0%|          | 1/902 [00:30<7:35:14, 30.32s/it]

Missing dominant segment that is centered at:974 (ms)
segment (926:1007) is isolated.
Missing dominant segment that is centered at:765 (ms)
segment (717:799) is isolated.


  0%|          | 2/902 [01:22<10:19:07, 41.28s/it]
Operation terminated by user during MulticueF0v14>zmultiCandAC (line 1268)


In MulticueF0v14>SourceInfobyMultiCues050111 (line 279)
[f02,pl2]=zmultiCandAC(lx,lagspec,betaAC,lagslAC,timeslAC);

In MulticueF0v14 (line 60)
[f0raw,vuv,auxouts,prm]=SourceInfobyMultiCues050111(x,fs,prmin);

In legacy_STRAIGHT_shift_f0_contour (line 3)
    f0raw = MulticueF0v14(x, fs);



TypeError: cannot unpack non-iterable NoneType object

In [135]:
{key:len(vals) for key,vals in trial_data.items()}

{'signal': 2,
 'speaker': 2,
 'speaker_sex': 2,
 'sentence_type': 2,
 'word_int': 2,
 'mixture_signal': 2,
 'cue_signal': 2,
 'cue_speaker': 2,
 'cue_word': 2,
 'distractor_signal': 2,
 '_original_distractor_timit_indices': 2,
 '_original_cue_timit_index': 2,
 'distractor_words': 2,
 'distractor_speakers': 2,
 'distractor_conditions': 2,
 'distractor_sex': 2,
 'snrs': 2,
 'word': 2,
 '_original_timit_index': 2,
 'source': 2,
 'sr': 2,
 'signal_length': 2,
 'sentence_id': 2,
 'dialect_region': 2,
 'data_split': 2}

In [134]:
model_data = pd.DataFrame.from_dict(trial_data)

In [118]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']
word_to_ix_map = {val:key for key,val in class_map.items()}



In [56]:
def get_ix_from_words(words):
    if not isinstance(words, list):
        return words
    else:
        return [word_to_ix_map[word] for word in words]

model_timit_df['distractor_word_ints'] = model_timit_df['distractor_words'].apply(get_ix_from_words)


In [57]:
# model_timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [58]:
# out_path = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/')

# model_timit_df.to_pickle(out_path / 'timit_attn_stim_for_model_all_targets.pdpkl')

In [3]:
# f_path = '/om2/user/imgriff/datasets/timit/attn_task_dataframes/timit_attn_stim_for_model_all_targets.pdpkl'
# timit_df = pd.read_pickle(f_path)


In [4]:
timit_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       '_original_distractor_timit_indices', '_original_cue_timit_index',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'word', '_original_timit_index', 'source',
       'sr', 'signal_length', 'sentence_id', 'dialect_region', 'data_split',
       'distractor_word_ints'],
      dtype='object')

In [6]:
from IPython.display import Audio


In [116]:
Audio(cue_inharm, rate=20000)

## Making Whisper Speech Dataset 


Same idea as the inharmonic but now with whisper speech, which does not require the min f0 computation.  

Use only single-distractors.  We are trying to emulate the concurent sentences experiment (experiment 5) in the Popham et al. 2018 paper. The paper doesn't state the SNR used in this experiement, so we will start with setting target and distractor to the same level (0dB). 

Proceedure for generating each set of trial stimuli:
 * select cue, target, and distractor signals
 * resynth all three signals as whispers 
 * mix target and distractor 
 * save all whisper signals (cue, target, distractor, and mixture)
 
 
#### Note:  could also include a few other levels for more data on performance (-3, +3 dB SNR) at a later data.


### Demo of generation for single trial 

In [102]:
target = target_timit['signal'].iloc[0]
speaker = target_timit['speaker'].iloc[0]

cue = cue_timit[cue_timit.speaker == speaker].sample(1)['signal'].item()

distractor_data = target_timit[target_timit.speaker!=speaker].sample(1)
distractor = distractor_data['signal'].item()

sr = 20_000

cue_whisper = make_whispered_speech(cue, sr, eng=eng)
target_whisper = make_whispered_speech(target, sr, eng=eng)
distractor_whisper = make_whispered_speech(distractor, sr, eng=eng)

Elapsed time is 0.323144 seconds.
Elapsed time is 0.020084 seconds.
Elapsed time is 0.156435 seconds.
Elapsed time is 0.445194 seconds.
Elapsed time is 0.324262 seconds.
Elapsed time is 0.019516 seconds.
Elapsed time is 0.166315 seconds.
Elapsed time is 0.443577 seconds.
Elapsed time is 0.335011 seconds.
Elapsed time is 0.021168 seconds.
Elapsed time is 0.253425 seconds.
Elapsed time is 0.437730 seconds.


### signals generated for all stim in scripts

Like the inharmonnic stimuli generation, all stimuli were made using `src/make_whisper_timit.py`, submitted as job arrays with `submit_make_whisper_timit.sh` 

## Make Clean only stim for normal speech 

In [2]:
all_timit = pd.read_pickle('/om2/user/imgriff/datasets/timit/attn_task_dataframes/timit_attn_stim_for_model_all_targets.pdpkl')

In [3]:
all_timit.head()

,signal,speaker,speaker_sex,sentence_type,word_int,mixture_signal,cue_signal,cue_speaker,cue_word,_original_distractor_timit_indices,...,snrs,word,_original_timit_index,source,sr,signal_length,sentence_id,dialect_region,data_split,distractor_word_ints
0,"[0.00012005048285174084, 0.0010252486123257909...",fdaw0,f,sx,552,"[0.09473495656716215, 0.07726425141867858, 0.0...","[-6.021130907082137e-05, 0.0002179752211393519...",fdaw0,slowed,[5748],...,-6,programs,15,train-dr1-fdaw0-sx146,20000,40000,146,dr1,train,[working]
1,"[0.011782246730868976, 0.08233421110820222, 0....",fdaw0,f,sx,461,"[-0.020054463556352937, 0.010147070949337084, ...","[1.686975271292467e-05, -0.0001628872552172854...",fdaw0,answer,[6095],...,-6,novel,17,train-dr1-fdaw0-sx326,20000,40000,326,dr1,train,[medical]
2,"[0.0026449234716838257, 0.009264113281959791, ...",fdml0,f,sx,646,"[0.002029729771053116, 0.004756987291048679, 0...","[0.00028486117086427246, -0.000579473807245304...",fdml0,tube,[978],...,-6,should,28,train-dr1-fdml0-sx429,20000,40000,429,dr1,train,[larger]
3,"[-0.00037831780339846264, -0.00045162531152865...",fecd0,f,sx,659,"[-0.018556469254813943, -0.007759291164789502,...","[-0.012412901495380104, -0.015420376258757239,...",fecd0,wound,[1679],...,-6,social,35,train-dr1-fecd0-sx158,20000,40000,158,dr1,train,[caused]
4,"[0.029901661481428206, 0.01862221753076525, 0....",fecd0,f,sx,393,"[0.12764520140291172, 0.0784103245386344, 0.02...","[-0.025108234638327256, -0.027701923768734996,...",fecd0,to,[342],...,-6,light,36,train-dr1-fecd0-sx248,20000,40000,248,dr1,train,[young]


In [29]:
clean_timit = all_timit.loc[(all_timit['distractor_conditions'] == 'ssn')
                            & (all_timit['snrs'] == 0)
                            ].reset_index(drop=True)

array([-0.0723974 , -0.08332606, -0.07563086, ..., -0.00727153,
       -0.00723461, -0.00474659])

In [33]:
clean_timit.to_pickle('/om2/user/imgriff/datasets/timit/clean_timit_targets_attn_task_0.1rms.pdpkl')

## Eg on making safe & nomralized timit 

In [ ]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'
timit_excerpts = pd.read_pickle(path)


In [43]:
## Made in timit_screen.ipynb on local desktop 

bad_sentences = ['sx74', 'sx144', 'sx376', 'si615', 'si670', 'si688', 'si693',
       'si917', 'si926', 'si927', 'si930', 'si931', 'si1078', 'si1102',
       'si1106', 'si1241', 'si1616', 'si1739', 'si1752', 'si1758',
       'si1767', 'si1933', 'si2020', 'si2040', 'si2064', 'si2112',
       'si2134', 'si2204', 'si2284']


timit_excerpts = timit_excerpts[~timit_excerpts.sentence_id.isin(bad_sentences)]

In [44]:
## Normalize all audio signals 
all_signals = np.stack(timit_excerpts.signal).astype('float32')

all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


all_signals = all_signals * 0.1 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

In [14]:
timit_excerpts.to_pickle('/om2/user/imgriff/datasets/timit/safe_sentences_timit_wsn_compatible_0.1rms.pdpkl')

NameError: name 'timit_excerpts' is not defined

In [88]:
timit_excerpts.columns

Index(['word', '_original_timit_index', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'],
      dtype='object')